# PyRosetta in the browser

This notebook runs PyRosetta entirely inside your browser. Python here is [Pyodide](https://pyodide.org), CPython compiled to WebAssembly, and PyRosetta is compiled to WebAssembly to run alongside it. No server runs any Python: the site only serves files, and what you compute stays in this tab.

Run the cells in order.

Install PyRosetta from the copy bundled with this site. The wheel is about 100 MB, so this takes a little while.

In [ ]:
%pip install pyrosetta

In [ ]:
import pyrosetta
pyrosetta.init()

Build a short peptide and score it with Rosetta's default score function, `ref2015`.

The first score takes about a minute. The first time anything uses the rotamer libraries in a session, Rosetta converts them to a faster binary form. Scoring again is quick.

In [ ]:
pose = pyrosetta.pose_from_sequence("AFILVWYGPSTNQ")
scorefxn = pyrosetta.get_score_function()
print(f"{pose.total_residue()} residues, ref2015 score {scorefxn(pose):.2f}")

Repack the side chains, then score the pose again.

In [ ]:
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.protocols.minimization_packing import PackRotamersMover

task = TaskFactory.create_packer_task(pose)
task.restrict_to_repacking()
PackRotamersMover(scorefxn, task).apply(pose)
print(f"after repacking: {scorefxn(pose):.2f}")

## Your own structures

The file browser on the left shows this notebook's folder, which is also Python's working directory. To work on a structure of your own, upload a PDB file with the file browser's upload button, then load it with `pyrosetta.pose_from_pdb("your-file.pdb")`.

Files that Python writes appear in the file browser too. Save the repacked peptide, then load it back. To keep a copy, right-click the file in the file browser and choose Download.

These files are kept in your browser's own storage, not on a server.

If Python cannot find a file you uploaded, reload the page and run the cells again. On a first visit the file browser can connect to Python too late, and in some private windows, Firefox's among them, the two never connect.

In [ ]:
pose.dump_pdb("repacked.pdb")
reloaded = pyrosetta.pose_from_pdb("repacked.pdb")
print(f"reloaded {reloaded.total_residue()} residues, ref2015 score {scorefxn(reloaded):.2f}")

## Structures from the Protein Data Bank

`pose_from_rcsb` downloads a structure from the RCSB Protein Data Bank by its ID. It downloads with Python's `urllib`, which cannot reach the network from a browser on its own, so `pyodide_http.patch_all()` first routes its requests through the browser.

This cell needs an internet connection. It saves two files next to this notebook: `1UBQ.pdb` as downloaded, and `1UBQ.clean.pdb` with only its `ATOM` and `TER` records, which is the one it loads.

In [ ]:
import pyodide_http
pyodide_http.patch_all()

from pyrosetta.toolbox.rcsb import pose_from_rcsb

ubiquitin = pose_from_rcsb("1UBQ")
print(f"1UBQ: {ubiquitin.total_residue()} residues")